# Qwen3.5-4B: priorización de tareas TI con LoRA y DoRA

**Pregunta:** ¿cómo cambian la calidad, el tiempo y la memoria al adaptar el mismo
checkpoint con LoRA y DoRA? Este notebook adapta el flujo del docente en
`main.ipynb` e incorpora aquí la generación de datos, validación y evaluación.
Los originales se conservan. No requiere ejecutar el notebook de evaluación del docente.

**Orden:** preparación y descarga → datos → modelo original → LoRA → DoRA → comparación.
Las tablas y conclusiones se calculan al ejecutar; no se presupone una mejora.

### Alcance acordado

Una ejecución por método, semilla 42, datos sintéticos en español. Se omiten el
diseño factorial, réplicas, 20% de ejemplos escritos por alumnos, regresión,
costos monetarios y PPT. Esto no satisface toda la rúbrica de `instructions.md`.
El checkpoint ya fue postentrenado por Qwen: «original» significa **sin nuestros
adaptadores**, no un modelo preentrenado sin instrucciones.

Todos los experimentos permanecen en este notebook; el script Slurm solo lo ejecuta.


## 1. Preparar Khipu y el entorno

Seguir `AGENTS.md`: el login admite instalación y transferencia; **no cargar el
modelo ni entrenar allí**. El experimento requiere un job GPU de Slurm.
No introducir tokens de Hugging Face en las celdas. Este modelo público no exige login.

`requerements.txt` es el archivo original: incluye `pywinpty` (Windows), un Torch
CUDA específico y no incluye Matplotlib ni scikit-learn. El nuevo `requirements.txt`
conserva las dependencias necesarias y evita trasladar ese entorno completo.

Preparación desde esta carpeta (Python 3.11):

```bash
# Este Python está en almacenamiento compartido; /usr/bin/python3.11 no existe en ds001.
KHIPU_PYTHON=/opt/ohpc/pub/apps/spack/local/linux-rocky8-zen2/aocc-4.2.0/python-3.11.7-ujz6vknvxczz5byyrbuykxbs7o4obmpw/bin/python3.11
"$KHIPU_PYTHON" -m venv .venv
.venv/bin/python -m pip install --upgrade pip
# CUDA 12.6: verificar compatibilidad con el driver del nodo GPU.
.venv/bin/python -m pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cu126
.venv/bin/python -m pip install -r requirements.txt
# Ejecuta solo la celda de descarga; no carga pesos en RAM/GPU.
.venv/bin/python - <<'PY'
import nbformat
n = nbformat.read('qwen35_4b_priorizacion.ipynb', as_version=4)
for c in n.cells:
    if 'download' in c.metadata.get('tags', []):
        exec(c.source)
PY
```

Después, `sbatch --time=00:30:00 run_qwen.sbatch smoke`; revisar el resultado antes de ejecutar
`sbatch run_qwen.sbatch full`. El wrapper solicita una RTX A6000, 4 CPUs y 48 GiB
de RAM. Smoke limita la ejecución a 30 minutos; full dispone de 4 horas. Revisar
disponibilidad y límites con `sinfo`, `sacctmgr` y `squeue` antes de enviar.
No se supone que una fracción GPU reserve una cantidad aislada de VRAM.

Para una sesión Jupyter interactiva, abrir este notebook con el kernel del entorno
**dentro de una asignación GPU** y ejecutar de arriba hacia abajo.


## 2. Descargar el checkpoint oficial (sin cargar el modelo)

Esta celda es autocontenida y se puede ejecutar por separado en el login.
La primera descarga resuelve `main` a un commit y lo registra en `models/model_manifest.json`.
Las siguientes reutilizan ese commit; no siguen automáticamente actualizaciones.
Se descargan los archivos del repositorio completo, incluidos pesos y tokenizer.


In [1]:
import json
import os
from pathlib import Path

if os.environ.get("QWEN_LOCAL_ONLY") != "1":
    # Standard HTTPS avoids the stalled Xet transport observed on this cluster.
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("HF_HOME", str(Path("models/hf-home").resolve()))
    from huggingface_hub import HfApi, snapshot_download

    model_id = "Qwen/Qwen3.5-4B"
    model_root = Path("models").resolve()
    model_root.mkdir(exist_ok=True)
    manifest_path = model_root / "model_manifest.json"
    previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    if previous and previous["model_id"] != model_id:
        raise ValueError("El manifiesto existente pertenece a otro checkpoint.")
    revision = previous["revision"] if previous else HfApi().model_info(model_id).sha
    snapshot = snapshot_download(
        repo_id=model_id, revision=revision, cache_dir=str(model_root / "hub"),
        max_workers=4,
    )
    manifest = {"model_id": model_id, "revision": revision,
                "snapshot_path": str(Path(snapshot).resolve())}
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print("Checkpoint disponible:", manifest)
else:
    print("Modo local: usar el checkpoint preparado antes de Slurm.")


Modo local: usar el checkpoint preparado antes de Slurm.


## 3. Configuración y trazabilidad

`QWEN_MODE=smoke` usa 16/8/8 ejemplos, dos pasos y la misma arquitectura y
adaptadores. Sus resultados se guardan separados y **no representan el experimento**.
`QWEN_MODE=prepare` verifica datos/métricas y genera su figura sin cargar el modelo.
El modo predeterminado es `full`: 400/100/100 ejemplos y tres épocas.

Las versiones efectivas, revisión, hardware, configuración, política y huella de
datos quedan registrados. No se activa WandB ni ningún servicio externo.


In [2]:
import gc
import hashlib
import importlib.metadata as metadata
import math
import platform
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, f1_score, recall_score

@dataclass
class Config:
    model_id: str = "Qwen/Qwen3.5-4B"
    seed: int = 42
    mode: str = os.environ.get("QWEN_MODE", "full")
    train_size: int = 400
    validation_size: int = 100
    test_size: int = 100
    rank: int = 16
    alpha: int = 32
    dropout: float = 0.0
    learning_rate: float = 2e-4
    epochs: int = 3
    batch_size: int = 1
    accumulation: int = 16
    max_seq_length: int = 512
    max_new_tokens: int = 128
    warmup_ratio: float = 0.03

config = Config()
if config.mode not in {"full", "smoke", "prepare"}:
    raise ValueError("QWEN_MODE debe ser full, smoke o prepare.")
if config.mode == "smoke":
    config.train_size, config.validation_size, config.test_size = 16, 8, 8
run_id = os.environ.get("QWEN_RUN_ID") or (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + config.mode
)
if Path(run_id).name != run_id:
    raise ValueError("QWEN_RUN_ID debe ser un nombre de carpeta, sin rutas.")
run_dir = Path("outputs") / run_id
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "figures").mkdir()
random.seed(config.seed)
np.random.seed(config.seed)

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False) + "\n", encoding="utf-8")

save_json(run_dir / "config.json", asdict(config))
packages = {p.metadata["Name"]: p.version for p in metadata.distributions()}
environment = {"python": sys.version, "platform": platform.platform(),
               "packages": packages, "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
               "status": "prepared", "mode": config.mode}
save_json(run_dir / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        check=True, capture_output=True, text=True)
(run_dir / "requirements-frozen.txt").write_text(freeze.stdout)
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "savefig.dpi": 160})
COLORS = {"Original": "#326A9F", "LoRA": "#A46E13", "DoRA": "#98658B"}
PRIORITIES = ["P1", "P2", "P3", "P4"]

def show_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(run_dir / "figures" / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Resultados:", run_dir.resolve())
display(pd.DataFrame([asdict(config)]).T.rename(columns={0: "valor"}))


Resultados: /home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/outputs/58318_smoke


,valor
model_id,Qwen/Qwen3.5-4B
seed,42
mode,smoke
train_size,16
validation_size,8
test_size,8
rank,16
alpha,32
dropout,0.0
learning_rate,0.0002


## 4. Política de prioridad y formato de respuesta

Evaluamos clasificación de **una tarea**, no ordenamiento de listas. La prioridad
depende de impacto, bloqueo, alternativa y plazo, no de mayúsculas o impaciencia.
Se aplica la primera regla satisfecha. Un plazo solo eleva prioridad si incumplirlo
tiene consecuencias operativas. El generador conserva los atributos usados para
auditar la etiqueta, pero el modelo recibe únicamente una descripción en español.

La respuesta debe ser exactamente `{"priority":"P2","reason":"..."}`.
La razón es una explicación breve; no se evalúa por coincidencia literal.


In [3]:
POLICY = (
    "Clasifica una tarea TI. Aplica la primera regla que corresponda: "
    "P1: incidente activo de seguridad o pérdida de datos, o servicio crítico "
    "caído sin alternativa. "
    "P2: trabajo bloqueado sin alternativa, o plazo con consecuencias operativas "
    "de hasta 24 horas. "
    "P3: degradación no crítica, alternativa disponible ante un problema, "
    "o plazo con consecuencias de hasta 7 días. "
    "P4: mantenimiento rutinario o mejoras opcionales sin impacto inmediato. "
    "La insistencia del solicitante no cambia estas reglas. "
    "Responde solo JSON con exactamente priority (P1/P2/P3/P4) y reason "
    "(explicación breve en español)."
)

def priority_for(a):
    if a["security"] or a["data_loss"] or (
        a["critical"] and a["outage"] and not a["workaround"]
    ):
        return "P1"
    if (a["blocked"] and not a["workaround"]) or (
        a["consequential"] and a["deadline_hours"] <= 24
    ):
        return "P2"
    if a["degradation"] or a["workaround"] or (
        a["consequential"] and a["deadline_hours"] <= 168
    ):
        return "P3"
    return "P4"

REASONS = {
    "P1": "Incidente activo de seguridad, pérdida de datos o caída crítica sin alternativa.",
    "P2": "Existe un bloqueo sin alternativa o un plazo operativo de hasta veinticuatro horas.",
    "P3": "El impacto es moderado, hay alternativa o el plazo operativo es de esta semana.",
    "P4": "Es mantenimiento o mejora opcional sin impacto inmediato en el trabajo.",
}
save_json(run_dir / "policy.json", {"instruction": POLICY, "reasons": REASONS})


## 5. Dataset sintético sin duplicados entre particiones

Construimos 600 ejemplos balanceados. Cada partición usa servicios y familias de
redacción diferentes; conservamos su identificación para auditar la separación.
Los atributos se seleccionan primero y la prioridad se deriva de la política.
Incluimos insistencia injustificada y tono tranquilo ante incidentes graves.

**Limitación:** los síntomas se construyen con un vocabulario controlado compartido.
Separar servicios y plantillas reduce la fuga literal, pero sigue siendo una prueba
de transferencia sintética; no demuestra generalización a tickets reales.


In [4]:
SERVICES = {
    "train": ["correo interno", "portal de matrículas", "inventario", "ventas", "nómina",
              "repositorio de código", "mesa de ayuda", "red de oficinas"],
    "validation": ["agenda de reservas", "sistema de almacén", "intranet", "impresión"],
    "test": ["facturación", "plataforma de cursos", "control de accesos", "compras"],
}
TEMPLATES = {
    "train": ["Servicio: {service}. Situación: {symptom} {deadline} {tone}",
              "Necesitamos revisar {service}: {symptom} {tone} {deadline}",
              "El equipo informa sobre {service}. {deadline} {symptom} {tone}",
              "Solicitud para {service}: {tone} {symptom} {deadline}"],
    "validation": ["Nota técnica de {service} — {symptom} Plazo: {deadline} {tone}",
                   "Área afectada: {service}. {tone} Estado observado: {symptom} {deadline}"],
    "test": ["Por favor evaluar la tarea en {service}. {deadline} Se observa: {symptom} {tone}",
             "Resumen operativo [{service}]: {tone} {deadline} Detalle: {symptom}"],
}
TONES = ["Por favor, revisarlo cuando corresponda.", "¡Es URGENTE, quiero atención ya!",
         "Lo comunico con calma; prioricen según el impacto.",
         "La persona solicitante insiste en que se atienda primero."]
BASE_ATTRIBUTES = dict(security=False, data_loss=False, critical=False, outage=False,
                       workaround=False, blocked=False, consequential=False,
                       deadline_hours=720, degradation=False)
SCENARIOS = [
    ("Se detectó extracción de credenciales en curso.", dict(security=True)),
    ("Se están borrando registros de producción sin autorización.", dict(data_loss=True)),
    ("El servicio crítico está completamente caído; no hay alternativa.",
     dict(critical=True, outage=True)),
    ("Un equipo no puede continuar su trabajo; no existe alternativa temporal.", dict(blocked=True)),
    ("Hay que entregar un cambio con consecuencias operativas si vence el plazo.",
     dict(consequential=True, deadline_hours=4)),
    ("El acceso de un área está bloqueado y no puede usar otro sistema.", dict(blocked=True)),
    ("La aplicación está lenta, pero permite trabajar; no es crítica.", dict(degradation=True)),
    ("El trabajo está bloqueado en la herramienta principal, pero hay una alternativa funcional.",
     dict(blocked=True, workaround=True)),
    ("Debe entregarse un cambio esta semana; incumplir afecta la operación.",
     dict(consequential=True, deadline_hours=72)),
    ("Se propone cambiar colores de la interfaz, sin impacto operativo.", {}),
    ("Se solicita ordenar documentación cuando haya tiempo; el trabajo continúa normal.", {}),
    ("Se desea una mejora opcional y no existe un compromiso operativo.", {}),
]

def make_dataset():
    rng = random.Random(config.seed)
    rows = []
    sizes = {"train": config.train_size, "validation": config.validation_size,
             "test": config.test_size}
    for split, size in sizes.items():
        assert size % 4 == 0
        pools = {p: [] for p in PRIORITIES}
        for template_id, template in enumerate(TEMPLATES[split]):
            for service in SERVICES[split]:
                for symptom, changes in SCENARIOS:
                    for tone in TONES:
                        a = {**BASE_ATTRIBUTES, **changes}
                        if a["consequential"]:
                            deadline = f"Vence en {a['deadline_hours']} horas y el plazo tiene consecuencias."
                        else:
                            deadline = "No hay un plazo adicional con consecuencias operativas."
                        text = template.format(service=service, symptom=symptom,
                                               deadline=deadline, tone=tone)
                        priority = priority_for(a)
                        pools[priority].append({"input": text, "priority": priority,
                            "reason": REASONS[priority], "attributes": a, "service": service,
                            "scenario_family": f"{split}_template_{template_id}",
                            "split": split, "provenance": "synthetic_policy_v1"})
        for priority, pool in pools.items():
            rng.shuffle(pool)
            assert len(pool) >= size // 4, (split, priority, len(pool))
            rows.extend(pool[:size // 4])
    rng.shuffle(rows)
    for row in rows:
        row["id"] = hashlib.sha256(row["input"].encode()).hexdigest()[:16]
    return rows

examples = make_dataset()
normalized = [" ".join(e["input"].lower().split()) for e in examples]
assert len(normalized) == len(set(normalized)), "Hay entradas duplicadas."
assert all(priority_for(e["attributes"]) == e["priority"] for e in examples)
splits = {s: [e for e in examples if e["split"] == s]
          for s in ["train", "validation", "test"]}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    for field in ["id", "scenario_family", "service"]:
        assert not ({e[field] for e in splits[a]} & {e[field] for e in splits[b]})
serialized = "".join(json.dumps(e, ensure_ascii=False, sort_keys=True) + "\n" for e in examples)
(run_dir / "dataset.jsonl").write_text(serialized, encoding="utf-8")
dataset_hash = hashlib.sha256(serialized.encode()).hexdigest()
save_json(run_dir / "dataset_manifest.json", {"sha256": dataset_hash,
          "seed": config.seed, "counts": {s: len(v) for s, v in splits.items()}})
distribution = pd.crosstab(pd.Series([e["priority"] for e in examples], name="Prioridad"),
                          pd.Series([e["split"] for e in examples], name="Partición"))
distribution.to_csv(run_dir / "class_distribution.csv")
display(distribution)
display(pd.DataFrame(examples)[["id", "split", "input", "priority"]].head(8))
ax = distribution[["train", "validation", "test"]].plot.bar(
    color=["#326A9F", "#A46E13", "#98658B"], rot=0)
ax.set(title="Ejemplos por prioridad y partición", xlabel="Prioridad", ylabel="Ejemplos")
ax.legend(title="Partición", loc="upper left", bbox_to_anchor=(1.02, 1))
show_figure(ax.figure, "class_distribution.png")


Partición,test,train,validation
Prioridad,,,
P1,2,4,2
P2,2,4,2
P3,2,4,2
P4,2,4,2


,id,split,input,priority
0,246a15146d49f968,test,Por favor evaluar la tarea en control de acces...,P2
1,4719d1515b5ccf18,test,Resumen operativo [plataforma de cursos]: Lo c...,P3
2,b2f186dbe884ba04,test,Resumen operativo [control de accesos]: La per...,P1
3,57710cf3dd1f7c5c,validation,Área afectada: impresión. Lo comunico con calm...,P3
4,1f9b00e3a17e8a85,train,El equipo informa sobre inventario. No hay un ...,P3
5,b7b7bd3a60cabc89,validation,Nota técnica de sistema de almacén — Un equipo...,P2
6,f298c5275651e676,train,Necesitamos revisar red de oficinas: Se detect...,P1
7,6ac3b58c15fa72e0,train,Necesitamos revisar portal de matrículas: Se p...,P4


## 6. Validación de respuestas y métricas

JSON válido no implica una prioridad correcta. Informamos por separado parsing,
esquema y calidad. El esquema exige exactamente dos claves y una razón no vacía.
JSON duplicando claves se rechaza. Markdown, texto extra o etiquetas `<think>`
no se limpian para mejorar artificialmente el resultado.

Una respuesta con esquema inválido se predice como `INVALID`; cuenta como error
en exactitud y como falso negativo en macro-F1 sobre las **cuatro clases reales**.
La matriz mantiene una quinta columna para esas respuestas.


In [5]:
def parse_response(raw):
    def unique_object(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"Clave duplicada: {key}")
            result[key] = value
        return result
    try:
        obj = json.loads(raw, object_pairs_hook=unique_object,
                         parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
    except (ValueError, TypeError) as exc:
        return {"json_ok": False, "schema_ok": False, "predicted": "INVALID",
                "error": str(exc), "reason": None}
    valid = (isinstance(obj, dict) and set(obj) == {"priority", "reason"}
             and isinstance(obj["priority"], str) and obj["priority"] in PRIORITIES
             and isinstance(obj["reason"], str) and bool(obj["reason"].strip()))
    return {"json_ok": True, "schema_ok": bool(valid),
            "predicted": obj["priority"] if valid else "INVALID",
            "error": None if valid else "Esquema inválido", "reason": obj.get("reason")
            if isinstance(obj, dict) else None}

def quality_metrics(rows):
    actual = [r["expected"] for r in rows]
    predicted = [r["predicted"] for r in rows]
    recall = recall_score(actual, predicted, labels=PRIORITIES, average=None, zero_division=0)
    return {"n_test": len(rows), "accuracy": float(accuracy_score(actual, predicted)),
            "macro_f1": float(f1_score(actual, predicted, labels=PRIORITIES,
                                      average="macro", zero_division=0)),
            "json_rate": float(np.mean([r["json_ok"] for r in rows])),
            "schema_rate": float(np.mean([r["schema_ok"] for r in rows])),
            **{f"recall_{p}": float(v) for p, v in zip(PRIORITIES, recall)}}

# Comprobaciones pequeñas: tipos inesperados y resultados inválidos no desaparecen.
assert parse_response('{"priority":"P1","reason":"Incidente"}')["schema_ok"]
for invalid in ['[]', 'null', '{"priority":[],"reason":"x"}',
                '{"priority":"P1","reason":""}',
                '```json\n{"priority":"P1","reason":"x"}\n```',
                '{"priority":"P1","priority":"P4","reason":"x"}']:
    assert parse_response(invalid)["predicted"] == "INVALID"
perfect = [{"expected": p, **parse_response(json.dumps({"priority": p, "reason": "x"}))}
           for p in PRIORITIES]
assert quality_metrics(perfect)["macro_f1"] == 1.0
wrong = [{"expected": p, **parse_response("no JSON")} for p in PRIORITIES]
assert quality_metrics(wrong)["accuracy"] == quality_metrics(wrong)["macro_f1"] == 0.0
assert priority_for({**BASE_ATTRIBUTES, "security": True, "workaround": True}) == "P1"
assert priority_for({**BASE_ATTRIBUTES, "consequential": True, "deadline_hours": 24}) == "P2"
assert priority_for({**BASE_ATTRIBUTES, "consequential": True, "deadline_hours": 168}) == "P3"
assert priority_for({**BASE_ATTRIBUTES, "consequential": True, "deadline_hours": 169}) == "P4"
print("Verificados: etiquetas, separación de datos, esquema y scoring de inválidos.")


Verificados: etiquetas, separación de datos, esquema y scoring de inválidos.


## 7. Cargar Qwen y preparar el chat

Usamos la clase nativa `Qwen3_5ForConditionalGeneration`, entrada de texto y
`enable_thinking=False`. No reutilizamos los marcadores Gemma del ejemplo.
Las tres condiciones comparten NF4 de cuatro bits, doble cuantización, precisión
y generación greedy. Nunca hacemos offload automático a CPU para esconder falta de VRAM.

Los parámetros originales quedan congelados. El conjunto de módulos adaptados
se restringe a las proyecciones MLP del lenguaje, no a visión ni a convoluciones.

El preflight exige al menos 12 GiB libres antes de cargar pesos. Es una
comprobación preventiva, no una garantía de memoria ni una reserva de VRAM.


In [6]:
GPU_ENABLED = config.mode != "prepare"
if GPU_ENABLED:
    import torch
    import bitsandbytes as bnb
    from datasets import Dataset
    from transformers import (AutoTokenizer, BitsAndBytesConfig,
        Qwen3_5ForConditionalGeneration, Trainer, TrainerCallback, TrainingArguments, set_seed)
    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

    if not os.environ.get("SLURM_JOB_ID"):
        raise RuntimeError("En Khipu ejecutar dentro de una asignación Slurm GPU, nunca en login.")
    if not torch.cuda.is_available():
        raise RuntimeError("El job no tiene CUDA disponible; revisar GPU y driver.")
    set_seed(config.seed)
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    manifest = json.loads(Path("models/model_manifest.json").read_text())
    if manifest["model_id"] != config.model_id:
        raise ValueError("Checkpoint incorrecto en el manifiesto.")
    snapshot_path = Path(manifest["snapshot_path"])
    if not snapshot_path.is_dir():
        raise FileNotFoundError("Preparar la descarga antes del job; ruta local inexistente.")
    tokenizer = AutoTokenizer.from_pretrained(snapshot_path, local_files_only=True)
    declared_tokens = json.loads((snapshot_path / "tokenizer_config.json").read_text())["added_tokens_decoder"]
    expected_eos = next(int(i) for i, v in declared_tokens.items()
                        if v["content"] == tokenizer.eos_token)
    assert tokenizer.eos_token_id == expected_eos, "Tokenizer incompleto o incompatible."
    assert len(tokenizer) > 200_000 and len(tokenizer.encode("Necesito revisar el sistema")) > 2
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    environment.update({"cuda_free_gib_at_start": free_bytes / 2**30,
                        "slurm_node": os.environ.get("SLURMD_NODENAME"),
                        "slurm_job_gpus": os.environ.get("SLURM_JOB_GPUS"),
                        "gpu": torch.cuda.get_device_name(0),
                        "gpu_total_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
                        "torch_cuda": torch.version.cuda, "dtype": str(compute_dtype),
                        "model": manifest, "dataset_sha256": dataset_hash, "status": "running"})
    save_json(run_dir / "environment.json", environment)
    print(environment["gpu"], compute_dtype, "commit", manifest["revision"])
    if free_bytes < 12 * 2**30:
        raise RuntimeError("Menos de 12 GiB VRAM libres. Solicitar una GPU dedicada; no se cargaron pesos.")

def render_prompt(text):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": POLICY}, {"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

def load_original(training=False):
    set_seed(config.seed)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        snapshot_path, local_files_only=True, quantization_config=quantization,
        dtype=compute_dtype, device_map={"": 0}, attn_implementation="sdpa")
    model.requires_grad_(False)
    if training:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        model.config.use_cache = False
        model.config.text_config.use_cache = False
    else:
        model.eval()
    return model

def release_memory():
    gc.collect()
    torch.cuda.empty_cache()

if GPU_ENABLED:
    example_prompt = render_prompt(splits["train"][0]["input"])
    assert "<start_of_turn>" not in example_prompt
    assert example_prompt.endswith("<think>\n\n</think>\n\n"), (
        "El template cambió: comprobar desactivación de thinking antes de continuar.")
    display(Markdown("**Prompt real de ejemplo:**"))
    print(example_prompt)


/home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA RTX A6000 torch.bfloat16 commit 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


**Prompt real de ejemplo:**

<|im_start|>system
Clasifica una tarea TI. Aplica la primera regla que corresponda: P1: incidente activo de seguridad o pérdida de datos, o servicio crítico caído sin alternativa. P2: trabajo bloqueado sin alternativa, o plazo con consecuencias operativas de hasta 24 horas. P3: degradación no crítica, alternativa disponible ante un problema, o plazo con consecuencias de hasta 7 días. P4: mantenimiento rutinario o mejoras opcionales sin impacto inmediato. La insistencia del solicitante no cambia estas reglas. Responde solo JSON con exactamente priority (P1/P2/P3/P4) y reason (explicación breve en español).<|im_end|>
<|im_start|>user
El equipo informa sobre inventario. No hay un plazo adicional con consecuencias operativas. La aplicación está lenta, pero permite trabajar; no es crítica. Lo comunico con calma; prioricen según el impacto.<|im_end|>
<|im_start|>assistant
<think>

</think>




## 8. Tokenización: pérdida solo en la respuesta

El prefijo de entrenamiento es exactamente el de inferencia. Después agregamos
el JSON esperado y el token de fin. Prompts y padding usan etiqueta `-100`.
El collator conserva estas etiquetas; no usa el collator del docente que vuelve
a calcularlas. Si una secuencia excede 512 tokens, fallamos en vez de truncarla.


In [7]:
def tokenize_row(example):
    prefix = tokenizer.encode(render_prompt(example["input"]), add_special_tokens=False)
    answer = json.dumps({"priority": example["priority"], "reason": example["reason"]},
                        ensure_ascii=False, separators=(",", ":"))
    suffix = tokenizer.encode(answer, add_special_tokens=False) + [tokenizer.eos_token_id]
    ids = prefix + suffix
    if len(ids) > config.max_seq_length:
        raise ValueError(f"Ejemplo {example['id']}: {len(ids)} tokens supera el límite; no se truncó.")
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prefix) + suffix}

def answer_collator(features):
    width = max(len(f["input_ids"]) for f in features)
    ids, masks, labels = [], [], []
    for f in features:
        pad = width - len(f["input_ids"])
        ids.append(f["input_ids"] + [tokenizer.pad_token_id] * pad)
        masks.append(f["attention_mask"] + [0] * pad)
        labels.append(f["labels"] + [-100] * pad)
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks),
            "labels": torch.tensor(labels)}

if GPU_ENABLED:
    tokenized = {s: [tokenize_row(e) for e in rows] for s, rows in splits.items()}
    for s, rows in tokenized.items():
        assert all(any(x != -100 for x in r["labels"]) for r in rows)
        print(s, "longitud máxima:", max(len(r["input_ids"]) for r in rows))
    train_dataset = Dataset.from_list(tokenized["train"])
    validation_dataset = Dataset.from_list(tokenized["validation"])
    pair = answer_collator(tokenized["train"][:2])
    assert torch.all(pair["labels"][pair["attention_mask"] == 0] == -100)
    first = tokenized["train"][0]
    answer_start = next(i for i, label in enumerate(first["labels"]) if label != -100)
    assert all(label == -100 for label in first["labels"][:answer_start])
    assert tokenizer.decode(first["labels"][answer_start:-1]).startswith('{"priority":')


train longitud máxima: 227
validation longitud máxima: 230
test longitud máxima: 232


## 9. Evaluación común y recursos

La latencia mide tokenización, generación y decodificación después de calentar el
modelo; sincronizamos CUDA antes/después. No incluye carga del checkpoint.
Los picos CUDA son memoria **asignada y reservada por este proceso**, no uso total
del dispositivo. Medimos entrenamiento e inferencia por separado.
Los adaptadores se evalúan recargados y sin fusionar. No usamos cachés especiales de DoRA.


In [8]:
all_predictions, all_metrics, training_histories = [], [], {}

def generate_one(model, text):
    torch.cuda.synchronize()
    started = time.perf_counter()
    inputs = tokenizer(render_prompt(text), return_tensors="pt", add_special_tokens=False).to("cuda:0")
    stop_ids = sorted({tokenizer.eos_token_id,
                       tokenizer.convert_tokens_to_ids("<|endoftext|>")})
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=config.max_new_tokens,
            do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_ids)
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    # Remove only the terminal stop token; keep unexpected special markers visible.
    visible_ids = new_tokens.tolist()
    if visible_ids and visible_ids[-1] in stop_ids:
        visible_ids = visible_ids[:-1]
    raw = tokenizer.decode(visible_ids, skip_special_tokens=False).strip()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return raw, elapsed, len(new_tokens)

def persist_results():
    if all_predictions:
        pd.DataFrame(all_predictions).to_csv(run_dir / "predictions.csv", index=False)
    if all_metrics:
        pd.DataFrame(all_metrics).to_csv(run_dir / "metrics.csv", index=False)
        save_json(run_dir / "metrics.json", all_metrics)

def evaluate(model, method, training_info=None):
    model.eval()
    model.config.use_cache = True
    model.config.text_config.use_cache = True
    generate_one(model, "La mejora del color del botón es opcional y no afecta la operación.")
    torch.cuda.reset_peak_memory_stats()
    rows = []
    for i, example in enumerate(splits["test"], 1):
        raw, latency, generated_tokens = generate_one(model, example["input"])
        rows.append({"method": method, "example_id": example["id"],
            "expected": example["priority"], "input": example["input"],
            "raw_output": raw, **parse_response(raw),
            "latency_seconds": latency, "generated_tokens": generated_tokens})
        if i % 10 == 0 or i == len(splits["test"]):
            print(method, i, "/", len(splits["test"]))
    latencies = [r["latency_seconds"] for r in rows]
    metrics = {"method": method, **quality_metrics(rows),
        "mean_latency_seconds": float(np.mean(latencies)),
        "median_latency_seconds": float(np.median(latencies)),
        "p95_latency_seconds": float(np.percentile(latencies, 95)),
        "inference_peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        "inference_peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
        "train_seconds": None, "train_peak_allocated_gib": None,
        "train_peak_reserved_gib": None, "trainable_parameters": 0,
        **(training_info or {})}
    all_predictions.extend(rows)
    all_metrics.append(metrics)
    persist_results()
    display(pd.DataFrame([metrics]))
    return metrics


## 10. Modelo original: antes de adaptar

Se utiliza el mismo test que para LoRA y DoRA. No seleccionamos hiperparámetros
mirando estas respuestas. Los resultados se guardan antes de iniciar entrenamiento.


In [9]:
if GPU_ENABLED:
    original = load_original()
    evaluate(original, "Original")
    del original
    release_memory()
else:
    print("prepare: datos y scoring verificados; no se midió ningún LLM.")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:11<2:14:29, 11.18s/it]

Loading weights:   1%|          | 6/723 [00:11<18:00,  1.51s/it]  

Loading weights:   1%|          | 8/723 [00:12<12:41,  1.07s/it]

Loading weights:   1%|          | 9/723 [00:12<10:39,  1.12it/s]

...

[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Original 8 / 8


,method,n_test,accuracy,macro_f1,json_rate,schema_rate,recall_P1,recall_P2,recall_P3,recall_P4,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,Original,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.654208,3.630594,4.173598,3.199261,3.261719,None,None,None,0


## 11. Entrenamiento compartido: LoRA y DoRA

LoRA aprende una actualización de bajo rango. DoRA agrega magnitudes entrenables
y usa LoRA para las direcciones. Usamos `use_dora=True` de PEFT, no una implementación
propia. **DoRA comienza desde Qwen original**, no desde el LoRA ya entrenado.

Los dos métodos usan r=16, alpha=32, dropout=0, lr=2e-4, acumulación 16 y tres
épocas; validamos/guardamos por época y elegimos menor pérdida de validación.
En smoke se usan exactamente dos pasos por método y se valida al terminar.
Comprobamos gradientes finitos no nulos y que al menos un parámetro cambia.

En Transformers 5.19, `warmup_steps` acepta una fracción cuando es menor que 1;
por eso se pasa 0.03 en ese argumento. Verificamos que equivale al 3% de pasos.


In [10]:
if GPU_ENABLED:
    class GradientAudit(TrainerCallback):
        def __init__(self):
            self.checked = False

        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
            grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            if not grads or not all(torch.isfinite(g).all().item() for g in grads):
                raise RuntimeError("Gradientes ausentes o no finitos.")
            if not self.checked and not any(torch.count_nonzero(g).item() for g in grads):
                raise RuntimeError("Todos los gradientes son cero antes de la primera actualización.")
            self.checked = True

def target_modules_for(model):
    targets = [name for name, layer in model.named_modules()
        if "language_model" in name.split(".") and ".mlp." in name
        and name.rsplit(".", 1)[-1] in {"gate_proj", "up_proj", "down_proj"}
        and isinstance(layer, (torch.nn.Linear, bnb.nn.Linear4bit))]
    if len(targets) != model.config.text_config.num_hidden_layers * 3:
        raise RuntimeError(f"Se esperaban tres proyecciones MLP por capa: {len(targets)} encontradas.")
    assert all("visual" not in name and "vision" not in name for name in targets)
    return targets

def train_adapter(method):
    release_memory()
    base = load_original(training=True)
    targets = target_modules_for(base)
    save_json(run_dir / f"{method.lower()}_targets.json", targets)
    adapter_config = LoraConfig(r=config.rank, lora_alpha=config.alpha,
        lora_dropout=config.dropout, target_modules=targets, bias="none",
        task_type="CAUSAL_LM", use_dora=(method == "DoRA"))
    model = get_peft_model(base, adapter_config)
    trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
    assert trainable and all("lora_" in name for name in trainable)
    # Una pequeña matriz B permite comprobar la actualización sin copiar el modelo.
    probe_name = next(n for n in trainable if "lora_B" in n)
    probe_before = trainable[probe_name].detach().cpu().clone()
    trainable_count = sum(p.numel() for p in trainable.values())
    model.print_trainable_parameters()
    smoke = config.mode == "smoke"
    args = TrainingArguments(
        output_dir=str(run_dir / "checkpoints" / method.lower()),
        num_train_epochs=config.epochs, max_steps=2 if smoke else -1,
        per_device_train_batch_size=config.batch_size, per_device_eval_batch_size=1,
        gradient_accumulation_steps=config.accumulation, learning_rate=config.learning_rate,
        weight_decay=0.01, warmup_steps=config.warmup_ratio, lr_scheduler_type="cosine",
        eval_strategy="steps" if smoke else "epoch", eval_steps=2 if smoke else None,
        save_strategy="steps" if smoke else "epoch", save_steps=2 if smoke else 500,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        save_total_limit=1, logging_steps=1 if smoke else 5, report_to="none",
        bf16=(compute_dtype == torch.bfloat16), fp16=(compute_dtype == torch.float16),
        optim="paged_adamw_8bit", gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        seed=config.seed, data_seed=config.seed, dataloader_num_workers=0,
        eval_accumulation_steps=1, prediction_loss_only=True)
    audit = GradientAudit()
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset,
        eval_dataset=validation_dataset, data_collator=answer_collator, callbacks=[audit])
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize()
    info = {"train_seconds": time.perf_counter() - started,
            "train_peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
            "train_peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
            "trainable_parameters": trainable_count}
    assert math.isfinite(result.training_loss) and audit.checked
    assert not torch.equal(probe_before, trainable[probe_name].detach().cpu()), "No hubo actualización."
    adapter_path = run_dir / "adapters" / method.lower()
    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(run_dir / f"{method.lower()}_training_history.csv", index=False)
    training_histories[method] = history
    save_json(run_dir / f"{method.lower()}_training.json", {
        **info, "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_loss": float(trainer.state.best_metric),
        "finite_nonzero_gradients": audit.checked, "parameter_updated": True,
        "optimizer_steps": trainer.state.global_step})
    del trainer, trainable, model, base
    release_memory()
    # Prueba de guardado/recarga y evaluación con un checkpoint base fresco.
    restored_base = load_original()
    restored = PeftModel.from_pretrained(restored_base, adapter_path, is_trainable=False)
    assert restored.peft_config["default"].use_dora == (method == "DoRA")
    evaluate(restored, method, info)
    del restored, restored_base
    release_memory()


## 12. Adaptar con LoRA

Primera adaptación, presupuesto fijado de antemano. El test no participa en la
selección del checkpoint. Se conserva el historial y se recarga el adaptador final.


In [11]:
if GPU_ENABLED:
    train_adapter("LoRA")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<01:48,  6.63it/s]

Loading weights:   2%|▏         | 12/723 [00:00<00:13, 54.42it/s]

Loading weights:   3%|▎         | 21/723 [00:00<00:10, 66.19it/s]

Loading weights:   4%|▍         | 29/723 [00:00<00:10, 66.97it/s]

...

Loading weights:  96%|█████████▌| 695/723 [00:07<00:00, 117.76it/s]

Loading weights:  98%|█████████▊| 710/723 [00:07<00:00, 125.55it/s]

Loading weights: 100%|██████████| 723/723 [00:07<00:00, 98.81it/s] 

LoRA 8 / 8


,method,n_test,accuracy,macro_f1,json_rate,schema_rate,recall_P1,recall_P2,recall_P3,recall_P4,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,LoRA,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.945243,1.888953,2.1855,5.655019,5.78125,69.958536,6.313916,6.857422,18087936


## 13. Adaptar con DoRA mediante PEFT

Mismos datos, GPU, precisión y presupuesto; cambia únicamente el método de
adaptación. Los parámetros de magnitud adicionales se incluyen en el conteo.


In [12]:
if GPU_ENABLED:
    train_adapter("DoRA")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<01:40,  7.20it/s]

Loading weights:   2%|▏         | 11/723 [00:00<00:13, 52.68it/s]

Loading weights:   3%|▎         | 20/723 [00:00<00:10, 67.97it/s]

Loading weights:   4%|▍         | 28/723 [00:00<00:10, 66.94it/s]

...

Loading weights:  96%|█████████▋| 696/723 [00:07<00:00, 125.28it/s]

Loading weights:  98%|█████████▊| 709/723 [00:07<00:00, 77.08it/s] 

Loading weights: 100%|██████████| 723/723 [00:07<00:00, 95.87it/s]

DoRA 8 / 8


,method,n_test,accuracy,macro_f1,json_rate,schema_rate,recall_P1,recall_P2,recall_P3,recall_P4,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,DoRA,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,18.560693,18.374208,21.818405,8.39462,8.873047,148.509809,8.702924,9.263672,18759680


## 14. Comparación de calidad y matrices de confusión

Las barras usan escala 0–100%. Cada matriz conserva las cuatro prioridades y
la columna `INVALID`; filas = referencia, columnas = predicción. No hay barras
de incertidumbre: solo se realizó una semilla por método.


In [13]:
if GPU_ENABLED:
    metrics_df = pd.DataFrame(all_metrics).set_index("method").loc[list(COLORS)]
    display(metrics_df[["n_test", "accuracy", "macro_f1", "json_rate", "schema_rate",
                        "recall_P1", "recall_P2", "recall_P3", "recall_P4"]])
    fig, ax = plt.subplots()
    x = np.arange(4)
    fields = ["accuracy", "macro_f1", "json_rate", "schema_rate"]
    for i, method in enumerate(COLORS):
        bars = ax.bar(x + (i - 1) * .24, metrics_df.loc[method, fields].to_numpy(dtype=float) * 100,
                      .24, label=method, color=COLORS[method])
        ax.bar_label(bars, fmt="%.1f", fontsize=8)
    ax.set(xticks=x, xticklabels=["Exactitud", "Macro-F1", "JSON válido", "Esquema"],
           ylim=(0, 112), ylabel="Porcentaje (%)",
           title=f"Calidad sobre {len(splits['test'])} tareas de prueba ({config.mode})")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    show_figure(fig, "quality_comparison.png")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    matrices = {}
    columns = PRIORITIES + ["INVALID"]
    for method in COLORS:
        rows = [r for r in all_predictions if r["method"] == method]
        matrix = np.array([[sum(r["expected"] == p and r["predicted"] == q for r in rows)
                            for q in columns] for p in PRIORITIES])
        assert matrix.sum() == len(splits["test"])
        matrices[method] = matrix
        pd.DataFrame(matrix, index=PRIORITIES, columns=columns).to_csv(
            run_dir / f"{method.lower()}_confusion.csv", index_label="expected")
    maximum = max(int(m.max()) for m in matrices.values())
    for ax, (method, matrix) in zip(axes, matrices.items()):
        im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(1, maximum))
        for (i, j), value in np.ndenumerate(matrix):
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > maximum / 2 else "black")
        ax.set(title=method, xticks=range(5), xticklabels=columns,
               yticks=range(4), yticklabels=PRIORITIES, xlabel="Predicción")
    axes[0].set_ylabel("Referencia")
    show_figure(fig, "confusion_matrices.png")


,n_test,accuracy,macro_f1,json_rate,schema_rate,recall_P1,recall_P2,recall_P3,recall_P4
method,,,,,,,,,
Original,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LoRA,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
DoRA,8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 15. Pérdidas y costo computacional

Las curvas muestran pérdida de entrenamiento y de validación; su escala no es
exactitud. Los tiempos y memoria se comparan por separado. El original no tiene
entrenamiento: esos campos son ausentes, no mediciones de cero segundos.


In [14]:
if GPU_ENABLED:
    fig, ax = plt.subplots()
    for method, history in training_histories.items():
        for field, style, label in [("loss", "-", "entrenamiento"),
                                    ("eval_loss", "--", "validación")]:
            if field in history:
                values = history.dropna(subset=[field])
                ax.plot(values["step"], values[field], style, marker="o",
                        color=COLORS[method], label=f"{method}: {label}")
    ax.set(xlabel="Paso del optimizador", ylabel="Pérdida (cross-entropy)",
           title="Evolución de la pérdida")
    ax.legend()
    show_figure(fig, "loss_curves.png")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    views = [("mean_latency_seconds", "Latencia media de inferencia", "Segundos / tarea", list(COLORS)),
             ("train_seconds", "Duración del entrenamiento", "Segundos", ["LoRA", "DoRA"]),
             ("train_peak_allocated_gib", "Pico asignado (entrenamiento)", "GiB", ["LoRA", "DoRA"])]
    for ax, (field, title, unit, methods) in zip(axes, views):
        bars = ax.bar(methods, [metrics_df.loc[m, field] for m in methods],
                      color=[COLORS[m] for m in methods])
        ax.bar_label(bars, fmt="%.2f", fontsize=9, padding=3)
        ax.set(title=title, ylabel=unit)
        ax.margins(y=.18)
    show_figure(fig, "resource_comparison.png")
    display(metrics_df[["trainable_parameters", "train_seconds", "mean_latency_seconds",
                        "p95_latency_seconds", "inference_peak_allocated_gib",
                        "inference_peak_reserved_gib", "train_peak_allocated_gib",
                        "train_peak_reserved_gib"]])


,trainable_parameters,train_seconds,mean_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_peak_allocated_gib,train_peak_reserved_gib
method,,,,,,,,
Original,0,NaN,3.654208,4.173598,3.199261,3.261719,NaN,NaN
LoRA,18087936,69.958536,1.945243,2.185500,5.655019,5.781250,6.313916,6.857422
DoRA,18759680,148.509809,18.560693,21.818405,8.394620,8.873047,8.702924,9.263672


## 16. Ejemplos, conclusiones y comprobación final

La explicación de la prioridad se revisa cualitativamente; no hay juez LLM.
La evaluación no obliga a que LoRA o DoRA mejoren. Una sola semilla y datos
sintéticos no permiten afirmar significancia, robustez ni superioridad universal.


In [15]:
if GPU_ENABLED:
    predictions_df = pd.DataFrame(all_predictions)
    wide = predictions_df.pivot(index="example_id", columns="method", values="predicted")
    changed = wide[(wide["Original"] != wide["LoRA"]) | (wide["LoRA"] != wide["DoRA"])]
    chosen_ids = changed.index[:3] if len(changed) else wide.index[:3]
    display(predictions_df[predictions_df.example_id.isin(chosen_ids)][
        ["method", "input", "expected", "predicted", "raw_output", "error"]])
    lines = ["### Resultados observados"]
    for method in COLORS:
        row = metrics_df.loc[method]
        lines.append(f"- {method}: exactitud {row.accuracy:.1%}, macro-F1 {row.macro_f1:.3f}, "
                     f"esquema válido {row.schema_rate:.1%}.")
    for method in ["LoRA", "DoRA"]:
        delta = metrics_df.loc[method, "macro_f1"] - metrics_df.loc["Original", "macro_f1"]
        lines.append(f"- Cambio macro-F1 de {method} frente al original: {delta:+.3f}.")
    delta = metrics_df.loc["DoRA", "macro_f1"] - metrics_df.loc["LoRA", "macro_f1"]
    lines.append(f"- Cambio DoRA frente a LoRA: {delta:+.3f}; interpretar junto al tiempo y VRAM.")
    if config.mode == "smoke":
        lines.append("**Solo smoke: estos valores verifican el pipeline, no son el experimento final.**")
    lines.append("Datos sintéticos, una semilla y una política determinista: validar con tickets "
                 "reales y réplicas antes de generalizar.")
    conclusion = "\n\n".join(lines)
    display(Markdown(conclusion))
    (run_dir / "conclusions.md").write_text(conclusion + "\n", encoding="utf-8")
    assert set(predictions_df.method) == set(COLORS)
    assert len(predictions_df) == len(splits["test"]) * 3
    assert not predictions_df.duplicated(["method", "example_id"]).any()
    persisted = pd.read_csv(run_dir / "predictions.csv", keep_default_na=False)
    for method in COLORS:
        recalculated = quality_metrics(persisted[persisted.method == method].to_dict("records"))
        assert np.isclose(recalculated["accuracy"], metrics_df.loc[method, "accuracy"])
        assert np.isclose(recalculated["macro_f1"], metrics_df.loc[method, "macro_f1"])
    environment["status"] = "completed"
    save_json(run_dir / "environment.json", environment)
else:
    environment["status"] = "data_checks_completed_no_llm_execution"
    save_json(run_dir / "environment.json", environment)
print("Artefactos guardados en", run_dir.resolve())


,method,input,expected,predicted,raw_output,error
0,Original,Por favor evaluar la tarea en control de acces...,P2,P2,"{\n ""priority"": ""P2"",\n ""reason"": ""La tarea ...",None
3,Original,Por favor evaluar la tarea en control de acces...,P1,P1,"{\n ""priority"": ""P1"",\n ""reason"": ""Se están ...",None
6,Original,Por favor evaluar la tarea en compras. No hay ...,P3,P3,"{\n ""priority"": ""P3"",\n ""reason"": ""La aplica...",None
8,LoRA,Por favor evaluar la tarea en control de acces...,P2,P2,"{""priority"":""P2"",""reason"":""Plazo con consecuen...",None
11,LoRA,Por favor evaluar la tarea en control de acces...,P1,P1,"{""priority"":""P1"",""reason"":""Borrado de datos de...",None
14,LoRA,Por favor evaluar la tarea en compras. No hay ...,P3,P3,"{""priority"":""P3"",""reason"":""Degradación no crít...",None
16,DoRA,Por favor evaluar la tarea en control de acces...,P2,P2,"{""priority"":""P2"",""reason"":""Plazo con consecuen...",None
19,DoRA,Por favor evaluar la tarea en control de acces...,P1,P1,"{""priority"":""P1"",""reason"":""Borrado de datos de...",None
22,DoRA,Por favor evaluar la tarea en compras. No hay ...,P3,P3,"{""priority"":""P3"",""reason"":""Degradación no crít...",None


### Resultados observados

- Original: exactitud 100.0%, macro-F1 1.000, esquema válido 100.0%.

- LoRA: exactitud 100.0%, macro-F1 1.000, esquema válido 100.0%.

- DoRA: exactitud 100.0%, macro-F1 1.000, esquema válido 100.0%.

- Cambio macro-F1 de LoRA frente al original: +0.000.

- Cambio macro-F1 de DoRA frente al original: +0.000.

- Cambio DoRA frente a LoRA: +0.000; interpretar junto al tiempo y VRAM.

**Solo smoke: estos valores verifican el pipeline, no son el experimento final.**

Datos sintéticos, una semilla y una política determinista: validar con tickets reales y réplicas antes de generalizar.

Artefactos guardados en /home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/outputs/58318_smoke


## Referencias y siguientes pasos

- [Checkpoint Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B)
- [Arquitectura Qwen3.5 en Transformers](https://huggingface.co/docs/transformers/model_doc/qwen3_5)
- [DoRA en PEFT](https://huggingface.co/docs/peft/package_reference/lora_variant_dora)
- [Chat template oficial](https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/chat_template.jinja)
- [Khipu: envío de jobs](https://docs.khipu.utec.edu.pe/guia-de-usuario/enviar-jobs/)

Para retomar la rúbrica completa: añadir datos escritos por el equipo y datos
reales, repetir semillas, ampliar el diseño factorial, investigar rendimientos
marginales y evaluar costo monetario. Mantener el test reservado para comparación
final. No volver a ajustar hiperparámetros usando los errores de este test.
